# Phase 3-5 - Rescue pipeline (Model 1 + Model 2 + tracking + memory + priority)

```
frame -> Model 1 detector -> ByteTrack -> camera-motion compensation
      -> group people -> memory (time stranded, water trend)
      -> Model 2 flood map (severity, isolation) -> risk 0-1 -> Critical/High/Medium/Low
      -> annotated video, priority map, victims.csv, alerts.json
```

Nothing is trained here. Needs on Drive (`MyDrive/FloodRescue/`):

| file | from |
|---|---|
| `models/flood_rehearsal.pt` | Phase 2b (already there) |
| `models/flood_severity_unet_scse.pt` | **upload** `outputs/model2/runs/unet_r50_scse_hybrid/best.pt` from the repo, renamed |
| `data/flood_c2a_coco.zip` | Phase 2 (already there) |
| `videos/*.mp4` | **upload** 1-3 real aerial flood clips (see section 8) |

`Runtime > Change runtime type > GPU` (T4 is fine). Skip section 6 (optional).
Section 8 runs every video in `videos/` (roughly 8-12 processed fps on a T4).
Full checklist: `docs/FINAL_DEMO_PLAN.md`.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
PROJECT = '/content/drive/MyDrive/FloodRescue'
MODELS, OUT = f'{PROJECT}/models', f'{PROJECT}/outputs/rescue'
DET = f'{MODELS}/flood_rehearsal.pt'
SEV = f'{MODELS}/flood_severity_unet_scse.pt'
C2A_ZIP = f'{PROJECT}/data/flood_c2a_coco.zip'
VIDEOS = f'{PROJECT}/videos'
os.makedirs(OUT, exist_ok=True); os.makedirs(VIDEOS, exist_ok=True)
for p in [DET, SEV, C2A_ZIP]:
    assert os.path.exists(p), f'missing {p}'
print('ok; videos on Drive:', [os.path.basename(v) for v in glob.glob(f'{VIDEOS}/*')])

## 3. Install

In [ ]:
!pip -q install "ultralytics>=8.3.0" "sahi>=0.11.20" "segmentation-models-pytorch>=0.3.3" pycocotools lapx
import ultralytics, sahi, segmentation_models_pytorch as smp, torch
print(ultralytics.__version__, sahi.__version__, smp.__version__, torch.cuda.get_device_name(0))

## 4. Write the code, run the logic tests

In [ ]:
!mkdir -p /content/rescue

In [ ]:
%%writefile /content/rescue/__init__.py
"""
Rescue pipeline (Phases 3-5): the two trained models joined into one system.

    frame -> detect (Model 1, optional SAHI) -> track (ByteTrack)
          -> camera-motion compensation -> group people into rescue units
          -> memory (time stranded, water trend) <- severity (Model 2, FloodNet)
          -> risk score in [0, 1] -> Critical / High / Medium / Low
          -> annotated video, priority map, victim table, alerts

Nothing in this package is trained. Entry point: scripts/run_rescue.py.
"""


In [ ]:
%%writefile /content/rescue/detect.py
"""
Module 1 - victim detection: Model 1 (YOLOv8, flood_rehearsal.pt), optionally with SAHI.

SAHI cuts the frame into overlapping tiles, runs the detector on each tile and
on the full frame, and merges everything with NMS. A person 15 px tall in the
frame becomes ~3x larger inside a tile, which is where plain YOLO loses them.

Merging uses NMS on IoU, not SAHI's default (greedy merge on intersection over
the smaller box): the default fuses neighbours in a crowd into one big box,
and the head count of a group matters for the risk score.
"""

import numpy as np
from ultralytics import YOLO

PERSON, VEHICLE, ANIMAL = 0, 1, 2
NAMES = {PERSON: "person", VEHICLE: "vehicle", ANIMAL: "animal"}


class Dets:
    """Detections in one frame. Indexable the way ByteTrack expects (conf / cls / xywh)."""

    def __init__(self, xyxy, conf, cls):
        self.xyxy = np.asarray(xyxy, np.float32).reshape(-1, 4)
        self.conf = np.asarray(conf, np.float32).reshape(-1)
        self.cls = np.asarray(cls, np.float32).reshape(-1)

    @property
    def xywh(self):
        x1, y1, x2, y2 = self.xyxy.T
        return np.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], 1)

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, i):
        return Dets(self.xyxy[i], self.conf[i], self.cls[i])


class Detector:
    def __init__(self, weights, imgsz=960, conf=0.25, device=None,
                 sahi=False, slice_size=512, overlap=0.2, nms_iou=0.5):
        self.imgsz, self.conf, self.sahi = imgsz, conf, sahi
        self.slice_size, self.overlap, self.nms_iou = slice_size, overlap, nms_iou
        self.device = device
        if sahi:
            from sahi import AutoDetectionModel
            self.model = AutoDetectionModel.from_pretrained(
                model_type="ultralytics", model_path=str(weights), confidence_threshold=conf,
                device=device or ("cuda:0" if _cuda() else "cpu"), image_size=imgsz)
        else:
            self.model = YOLO(str(weights))

    def __call__(self, frame_bgr) -> Dets:
        if not self.sahi:
            r = self.model.predict(frame_bgr, imgsz=self.imgsz, conf=self.conf, device=self.device,
                                   max_det=1000, verbose=False)[0].boxes.cpu().numpy()
            return Dets(r.xyxy, r.conf, r.cls)

        from sahi.predict import get_sliced_prediction
        res = get_sliced_prediction(
            np.ascontiguousarray(frame_bgr[:, :, ::-1]), self.model,      # SAHI wants RGB
            slice_height=self.slice_size, slice_width=self.slice_size,
            overlap_height_ratio=self.overlap, overlap_width_ratio=self.overlap,
            perform_standard_pred=True, postprocess_type="NMS",
            postprocess_match_metric="IOU", postprocess_match_threshold=self.nms_iou,
            verbose=0)
        p = res.object_prediction_list
        if not p:
            return Dets(np.zeros((0, 4)), [], [])
        return Dets([o.bbox.to_xyxy() for o in p], [o.score.value for o in p],
                    [o.category.id for o in p])


def _cuda():
    import torch
    return torch.cuda.is_available()


In [ ]:
%%writefile /content/rescue/track.py
"""
Module 2 - tracking: ByteTrack (the implementation bundled with Ultralytics).

Not trained. ByteTrack matches boxes frame to frame with a Kalman filter and
IoU, in two passes: confident detections first, then low-confidence ones,
which is what keeps a half-submerged person tracked when their score dips.
Running it here (instead of model.track) lets it take SAHI detections too.
"""

from types import SimpleNamespace

import numpy as np
from ultralytics.trackers.byte_tracker import BYTETracker

from .detect import Dets


class Tracker:
    def __init__(self, fps=30.0, high=0.25, low=0.1, new=0.25, buffer_s=3.0, match=0.8):
        frames = max(1, int(round(buffer_s * fps)))
        args = SimpleNamespace(tracker_type="bytetrack", track_high_thresh=high, track_low_thresh=low,
                               new_track_thresh=new, track_buffer=frames,
                               match_thresh=match, fuse_score=True)
        self.bt = BYTETracker(args)
        # Older Ultralytics rescales the buffer by frame_rate/30; pin it to frames.
        self.bt.max_time_lost = self.bt.max_frames_lost = frames

    def update(self, dets: Dets, frame):
        """Returns confirmed tracks as an (N, 7) array: x1, y1, x2, y2, track_id, score, cls."""
        out = np.asarray(self.bt.update(dets, frame), np.float32)
        return out[:, :7] if out.ndim == 2 and len(out) else np.zeros((0, 7), np.float32)


In [ ]:
%%writefile /content/rescue/motion.py
"""
Camera-motion compensation.

A drone drifts and pans. In raw pixels a person sitting still on a roof looks
like they are moving, and the water around them seems to change because the
view changed. So every frame is registered to the first one: corners are
tracked with Lucas-Kanade optical flow and a similarity transform (shift,
rotation, scale) is fitted with RANSAC. Chaining those gives a transform from
the current frame into "scene" coordinates (the first frame), in which
stillness and the water trend are measured.

Valid for a mostly flat scene seen from above, which is what a flood plain
from drone height is. It fails on fast pans with little texture (open water);
then the last good transform is kept and the frame is counted as unreliable.
"""

import cv2
import numpy as np


class CameraMotion:
    def __init__(self, enabled=True, work_width=640, max_corners=400, min_inliers=30):
        self.enabled, self.work_width = enabled, work_width
        self.max_corners, self.min_inliers = max_corners, min_inliers
        self.prev = None
        self.scale = 1.0
        self.T = np.eye(3)            # current frame -> scene (first frame)
        self.failures = 0

    def update(self, frame_bgr):
        if not self.enabled:
            return self.T
        h, w = frame_bgr.shape[:2]
        self.scale = min(1.0, self.work_width / w)
        g = cv2.cvtColor(cv2.resize(frame_bgr, None, fx=self.scale, fy=self.scale), cv2.COLOR_BGR2GRAY)
        if self.prev is not None:
            step = self._step(self.prev, g)
            if step is None:
                self.failures += 1
            else:
                S = np.diag([self.scale, self.scale, 1.0])
                # step maps current -> previous in the downscaled image; lift to full res
                self.T = self.T @ (np.linalg.inv(S) @ step @ S)
        self.prev = g
        return self.T

    def _step(self, prev, cur):
        p0 = cv2.goodFeaturesToTrack(prev, self.max_corners, 0.01, 8)
        if p0 is None or len(p0) < self.min_inliers:
            return None
        p1, st, _ = cv2.calcOpticalFlowPyrLK(prev, cur, p0, None)
        ok = st.reshape(-1) == 1
        if ok.sum() < self.min_inliers:
            return None
        M, inl = cv2.estimateAffinePartial2D(p1[ok], p0[ok], method=cv2.RANSAC, ransacReprojThreshold=2.0)
        if M is None or inl is None or inl.sum() < self.min_inliers:
            return None
        return np.vstack([M, [0, 0, 1]])

    def to_scene(self, pts):
        pts = np.asarray(pts, np.float64).reshape(-1, 2)
        return pts @ self.T[:2, :2].T + self.T[:2, 2]

    @property
    def zoom(self):
        """Scene pixels per frame pixel (<1 after the camera zooms in or descends)."""
        return float(np.sqrt(abs(np.linalg.det(self.T[:2, :2]))))


In [ ]:
%%writefile /content/rescue/severity.py
"""
Module 4 - flood severity: Model 2 (U-Net R50 + scSE, trained on FloodNet).

The segmenter labels every pixel with one of FloodNet's 10 classes. Two
numbers are read from that map around each rescue group:

  severity   share of FLOODED pixels (water, flooded road, flooded building)
             in a disk around the group, ignoring background. 0 = dry
             surroundings, 1 = surrounded by water. It says how much water
             is around the group, NOT how deep it is - depth cannot be read
             from a single RGB image.
  isolation  distance from the group to the nearest "mainland": dry ground
             (dry road, grass, dry building, unlabelled background) that is
             connected to the edge of the view, i.e. leads somewhere. As a
             fraction of `isolation_full` x frame diagonal, capped at 1.
             On mainland -> 0. On a dry island or a roof surrounded by
             water -> grows with the water between them and dry land.
             No mainland anywhere in view -> 1.

FloodNet is nadir imagery from ~200 ft after Hurricane Harvey. On oblique or
very different footage the map gets worse; check the overlay before trusting
the numbers.
"""

import cv2
import numpy as np
import torch
import torch.nn.functional as F

CLASSES = ["background", "building-flooded", "building-non-flooded", "road-flooded",
           "road-non-flooded", "water", "tree", "vehicle", "pool", "grass"]
FLOODED = [CLASSES.index(c) for c in ("building-flooded", "road-flooded", "water")]
DRY = [CLASSES.index(c) for c in ("background", "building-non-flooded", "road-non-flooded", "grass")]
BACKGROUND = 0
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
PALETTE = np.array([[0, 0, 0], [255, 0, 0], [180, 120, 120], [160, 150, 20], [140, 140, 140],
                    [61, 230, 250], [0, 82, 255], [255, 0, 245], [255, 235, 0], [4, 250, 7]],
                   np.uint8)   # RGB, same as floodnet_train.py


def build_model(cfg):
    import segmentation_models_pytorch as smp
    kw = dict(encoder_name=cfg["encoder"], encoder_weights=None, classes=len(CLASSES))
    if cfg["arch"] == "Unet":
        return smp.Unet(decoder_attention_type=cfg["attention"], **kw)
    return getattr(smp, cfg["arch"])(**kw)


class SeverityModel:
    def __init__(self, ckpt_path, device=None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if ck.get("classes", CLASSES) != CLASSES:
            raise SystemExit(f"{ckpt_path}: class list differs from FloodNet's")
        self.model = build_model(ck["config"]).to(self.device).eval()
        self.model.load_state_dict(ck["model"])
        self.frame = tuple(ck.get("frame", (1024, 768)))     # (W, H) it was trained at
        self.half = self.device.type == "cuda"
        if self.half:
            self.model.half()
        self.run = ck.get("run", "?")

    @torch.no_grad()
    def segment(self, frame_bgr):
        """Class-id map (H, W) uint8 at the frame's own resolution."""
        h, w = frame_bgr.shape[:2]
        W, H = self.frame
        if (h > w) != (H > W):            # portrait footage: segment it as landscape
            W, H = H, W
        x = cv2.resize(frame_bgr, (W, H), interpolation=cv2.INTER_AREA)[:, :, ::-1]
        x = (x.astype(np.float32) / 255.0 - MEAN) / STD
        x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).to(self.device)
        x = x.half() if self.half else x
        logits = self.model(x)
        pred = logits.argmax(1)[0].byte().cpu().numpy()
        return cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)


class SeverityMap:
    """Precomputed lookups on one segmentation, so each group costs O(disk area)."""

    def __init__(self, seg, isolation_full=0.25):
        self.seg = seg
        self.flooded = np.isin(seg, FLOODED)
        self.valid = seg != BACKGROUND
        diag = float(np.hypot(*seg.shape))
        self.norm = isolation_full * diag
        main = self._mainland(seg, diag)
        # distance from every pixel to the nearest mainland pixel
        self.dist = cv2.distanceTransform(1 - main, cv2.DIST_L2, 5) if main.any() else None
        self.counts = np.bincount(seg.ravel(), minlength=len(CLASSES))

    @staticmethod
    def _mainland(seg, diag):
        dry = np.isin(seg, DRY).astype(np.uint8)
        k = max(3, int(diag * 0.01) | 1)          # bridge people, cars, canopy over a road
        dry = cv2.morphologyEx(dry, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
        n, lab = cv2.connectedComponents(dry, connectivity=8)
        edge = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
        return np.isin(lab, edge[edge > 0]).astype(np.uint8)

    def severity(self, cx, cy, radius):
        h, w = self.seg.shape
        r = int(max(4, radius))
        x0, x1 = max(0, int(cx) - r), min(w, int(cx) + r + 1)
        y0, y1 = max(0, int(cy) - r), min(h, int(cy) + r + 1)
        if x0 >= x1 or y0 >= y1:
            return 0.0
        yy, xx = np.ogrid[y0:y1, x0:x1]
        disk = (xx - cx) ** 2 + (yy - cy) ** 2 <= r * r
        valid = self.valid[y0:y1, x0:x1] & disk
        n = valid.sum()
        return float((self.flooded[y0:y1, x0:x1] & valid).sum() / n) if n else 0.0

    def isolation(self, cx, cy):
        if self.dist is None:
            return 1.0
        h, w = self.seg.shape
        d = self.dist[min(h - 1, max(0, int(cy))), min(w - 1, max(0, int(cx)))]
        return float(min(1.0, d / self.norm))

    def flooded_share(self):
        n = self.counts[1:].sum()
        return float(self.counts[FLOODED].sum() / n) if n else 0.0


def colorize(seg):
    """BGR colour image of a class map."""
    return PALETTE[seg][:, :, ::-1].copy()


In [ ]:
%%writefile /content/rescue/memory.py
"""
Module 3 - short-term memory, and grouping people into rescue units.

Memory keeps a small state per tracked person (by ByteTrack ID), in scene
coordinates (see motion.py), so a single frame is never the whole story:

  stranded_s   seconds since the person last moved more than `move_factor`
               body sizes. Someone who has not moved for minutes is more
               likely stuck than someone walking out.
  trend        slope of the flood severity around them, in severity units
               per minute, over the last `trend_window_s` seconds. Positive
               = the water around them is rising (or spreading).
  seen_s       how long they have been in view.

Re-identification by place: ByteTrack gives a person a new ID when their
detection drops out for longer than its buffer (common for a half-submerged
person). In scene coordinates a still person stays at the same spot, so a
NEW track that appears within `move_factor` body sizes of a track that was
lost earlier inherits that track's memory (time stranded, water history)
instead of starting from zero.

A track that has been gone longer than `ttl_s` is forgotten. Not trained.
"""

from collections import deque

import numpy as np


class TrackState:
    __slots__ = ("first_t", "last_t", "anchor", "still_since", "size", "sev", "pos")

    def __init__(self, t, pos, size):
        self.first_t = self.last_t = self.still_since = t
        self.anchor = np.asarray(pos, np.float64)
        self.pos = self.anchor.copy()
        self.size = size
        self.sev = deque(maxlen=512)            # (t, severity)


class Memory:
    def __init__(self, move_factor=1.0, trend_window_s=60.0, trend_min_s=10.0, ttl_s=10.0):
        self.move_factor, self.trend_window_s = move_factor, trend_window_s
        self.trend_min_s, self.ttl_s = trend_min_s, ttl_s
        self.tracks: dict[int, TrackState] = {}

    def observe_frame(self, t, ids, scene_pos, scene_size):
        """All people of one frame. Known tracks first, so a new track can only
        inherit from a track that is NOT in this frame."""
        ids = [int(i) for i in ids]
        new = [k for k, tid in enumerate(ids) if tid not in self.tracks]
        for k, tid in enumerate(ids):
            if tid in self.tracks:
                self.observe(tid, t, scene_pos[k], scene_size[k])
        present = set(ids)
        for k in new:
            pos = np.asarray(scene_pos[k], np.float64)
            lost = [(np.linalg.norm(s.pos - pos), old) for old, s in self.tracks.items()
                    if old not in present and s.last_t < t
                    and np.linalg.norm(s.pos - pos) <= self.move_factor * max(s.size, scene_size[k])]
            if lost:
                old = min(lost)[1]
                self.tracks[ids[k]] = self.tracks.pop(old)
                present.add(old)
            self.observe(ids[k], t, pos, scene_size[k])

    def observe(self, tid, t, scene_pos, scene_size):
        s = self.tracks.get(tid)
        if s is None:
            self.tracks[tid] = TrackState(t, scene_pos, scene_size)
            return
        s.last_t, s.pos = t, np.asarray(scene_pos, np.float64)
        s.size = 0.8 * s.size + 0.2 * scene_size           # smooth box jitter
        if np.linalg.norm(s.pos - s.anchor) > self.move_factor * s.size:
            s.anchor, s.still_since = s.pos.copy(), t

    def observe_severity(self, tid, t, value):
        if tid in self.tracks:
            self.tracks[tid].sev.append((t, value))

    def forget(self, t):
        for tid in [k for k, s in self.tracks.items() if t - s.last_t > self.ttl_s]:
            del self.tracks[tid]

    def stranded_s(self, tid, t):
        s = self.tracks.get(tid)
        return t - s.still_since if s else 0.0

    def seen_s(self, tid, t):
        s = self.tracks.get(tid)
        return t - s.first_t if s else 0.0

    def trend_per_min(self, tid, t):
        s = self.tracks.get(tid)
        if s is None:
            return 0.0
        pts = [(ti, v) for ti, v in s.sev if t - ti <= self.trend_window_s]
        if len(pts) < 3 or pts[-1][0] - pts[0][0] < self.trend_min_s:
            return 0.0
        ts, vs = np.array(pts).T
        return float(np.polyfit(ts, vs, 1)[0] * 60.0)


class Group:
    """People close enough together to be rescued as one unit."""

    def __init__(self, members, boxes):
        self.members = members                          # track ids
        self.boxes = boxes                              # (n, 4) frame xyxy
        cx = (boxes[:, 0] + boxes[:, 2]) / 2
        cy = (boxes[:, 1] + boxes[:, 3]) / 2
        self.center = np.array([cx.mean(), cy.mean()])
        sizes = np.maximum(boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1])
        self.person_size = float(np.median(sizes))
        self.spread = float(np.max(np.hypot(cx - self.center[0], cy - self.center[1])))
        self.gid = min(members)
        self.features, self.risk, self.level = {}, 0.0, "Low"

    @property
    def count(self):
        return len(self.members)

    @property
    def bbox(self):
        return np.array([self.boxes[:, 0].min(), self.boxes[:, 1].min(),
                         self.boxes[:, 2].max(), self.boxes[:, 3].max()])


def group_people(tracks, link_factor=3.0, max_extent=12.0):
    """Single-linkage clustering: two people join when their centres are closer than
    link_factor x their mean body size. tracks: (N, 7) rows x1 y1 x2 y2 id score cls.

    Single linkage chains through a crowd, so a cluster wider than max_extent
    body sizes is split in two at the median of its longest axis, repeatedly:
    one rescue unit is people one boat or team can reach together."""
    n = len(tracks)
    if n == 0:
        return []
    b = tracks[:, :4]
    c = np.stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2], 1)
    size = np.maximum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1])
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    d = np.hypot(c[:, None, 0] - c[None, :, 0], c[:, None, 1] - c[None, :, 1])
    link = link_factor * (size[:, None] + size[None, :]) / 2
    for i, j in zip(*np.nonzero(np.triu(d <= link, 1))):
        parent[find(i)] = find(j)
    clusters = {}
    for i in range(n):
        clusters.setdefault(find(i), []).append(i)

    out, todo = [], [np.array(v) for v in clusters.values()]
    while todo:
        idx = todo.pop()
        pts = c[idx]
        span = pts.max(0) - pts.min(0)
        if len(idx) < 2 or span.max() <= max_extent * np.median(size[idx]):
            out.append(idx)
            continue
        ax = int(np.argmax(span))
        order = idx[np.argsort(pts[:, ax], kind="stable")]
        half = len(order) // 2
        todo += [order[:half], order[half:]]
    return [Group([int(tracks[i, 4]) for i in idx], b[idx]) for idx in out]


In [ ]:
%%writefile /content/rescue/risk.py
"""
Module 5 - risk and priority: a weighted formula, not a trained model.

    risk = sum_k w_k * f_k / sum_k w_k        (over the factors that are enabled)

Every factor f_k is scaled to [0, 1]:

  size       people in the group / `size_full`            more lives at one stop
  severity   flooded share around the group (Model 2)     how much water is around them
  stranded   seconds not moving / `stranded_full_s`       stuck, not walking out (memory)
  isolation  water between them and dry land (Model 2)  how hard they are to reach
  trend      rise in severity per minute / `trend_full`   water rising around them (memory)

Levels follow the project document: Critical > 0.8, High 0.5-0.8,
Medium 0.3-0.5, Low < 0.3.

Ablations (the planned "with vs without" comparisons) switch a module off
and re-normalise the remaining weights:
  no memory    -> stranded and trend dropped
  no severity  -> severity, isolation and trend dropped (all come from Model 2)

The weights are a design choice, stated as such - there is no ground truth
to fit them to. They put the water first (severity + trend + isolation = 0.55)
because that is what turns a group of people into an emergency.
"""

WEIGHTS = dict(size=0.20, severity=0.30, stranded=0.25, isolation=0.10, trend=0.15)
SCALE = dict(size_full=10, stranded_full_s=300.0, trend_full=0.10)
LEVELS = [(0.8, "Critical"), (0.5, "High"), (0.3, "Medium"), (0.0, "Low")]
MEMORY_FACTORS = {"stranded", "trend"}
SEVERITY_FACTORS = {"severity", "isolation", "trend"}


def factors(count, severity=0.0, stranded_s=0.0, isolation=0.0, trend_per_min=0.0, scale=SCALE):
    clip = lambda v: float(min(1.0, max(0.0, v)))
    return dict(size=clip(count / scale["size_full"]),
                severity=clip(severity),
                stranded=clip(stranded_s / scale["stranded_full_s"]),
                isolation=clip(isolation),
                trend=clip(trend_per_min / scale["trend_full"]))


def score(f, use_memory=True, use_severity=True, weights=WEIGHTS):
    off = (set() if use_memory else MEMORY_FACTORS) | (set() if use_severity else SEVERITY_FACTORS)
    w = {k: v for k, v in weights.items() if k not in off}
    total = sum(w.values())
    return sum(w[k] * f[k] for k in w) / total if total else 0.0


def level(risk):
    for th, name in LEVELS:
        if risk > th or th == 0.0:
            return name


In [ ]:
%%writefile /content/rescue/render.py
"""
Outputs: annotated frame + priority panel (the "rescue-priority map"), drawn with OpenCV.

Positions are image positions (pixels), not GPS - the datasets carry no drone
telemetry. With GPS + camera pose, the same group centres would project to
lat/long; that is a deployment step, noted in the project document.
"""

import cv2
import numpy as np

from .severity import colorize

LEVEL_BGR = {"Critical": (40, 40, 230), "High": (0, 140, 255), "Medium": (0, 215, 255), "Low": (80, 190, 60)}
VEHICLE_BGR = (230, 200, 40)
FONT = cv2.FONT_HERSHEY_SIMPLEX


def _text(img, s, org, scale, color, thick=1, bg=None):
    (tw, th), base = cv2.getTextSize(s, FONT, scale, thick)
    x, y = int(org[0]), int(org[1])
    if bg is not None:
        cv2.rectangle(img, (x - 2, y - th - 3), (x + tw + 2, y + base), bg, -1)
    cv2.putText(img, s, (x, y), FONT, scale, color, thick, cv2.LINE_AA)


def draw_frame(frame, persons, others, groups, seg=None, overlay_alpha=0.35, t=None):
    img = frame.copy()
    h, w = img.shape[:2]
    if seg is not None and overlay_alpha > 0:
        img = cv2.addWeighted(img, 1 - overlay_alpha, colorize(seg), overlay_alpha, 0)
    s = max(0.45, w / 2200)
    lw = max(1, int(round(w / 900)))

    for x1, y1, x2, y2, *_ in others:
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), VEHICLE_BGR, lw)
    level_of = {tid: g.level for g in groups for tid in g.members}
    for x1, y1, x2, y2, tid, *_ in persons:
        c = LEVEL_BGR[level_of.get(int(tid), "Low")]
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), c, lw)

    for rank, g in enumerate(groups, 1):
        c = LEVEL_BGR[g.level]
        pad = 0.5 * g.person_size
        x1, y1, x2, y2 = g.bbox + np.array([-pad, -pad, pad, pad])
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), c, lw + 1)
        label_y = y1 - 4 if y1 - 4 > 22 * s / 0.45 else y2 + 20 * s / 0.45   # below the box at the top edge
        _text(img, f"#{rank} {g.level} {g.risk:.2f} ({g.count})", (max(2, x1), label_y), s, (255, 255, 255),
              max(1, lw - 1), bg=c)

    n_people = int(sum(g.count for g in groups))
    hud = f"people {n_people}   groups {len(groups)}"
    if t is not None:
        hud = f"t={t:6.1f}s   " + hud
    _text(img, hud, (10, h - 10), s * 1.1, (255, 255, 255), lw, bg=(30, 30, 30))
    return img


def priority_panel(img, groups, title="Rescue priority", width=None, max_rows=12, use_memory=True):
    """Appends a ranked table to the right of the image."""
    h, w = img.shape[:2]
    pw = width or max(380, int(w * 0.32))
    panel = np.full((h, pw, 3), 28, np.uint8)
    s = pw / 700
    y = int(40 * s / 0.55)
    _text(panel, title, (14, y), s * 1.15, (255, 255, 255), 2)
    y += int(36 * s / 0.55)
    cols = [("#", 0.0), ("level", 0.06), ("risk", 0.30), ("ppl", 0.42), ("flood", 0.53),
            ("stuck", 0.67), ("isol.", 0.82)]
    for name, fx in cols:
        _text(panel, name, (32 + fx * (pw - 40), y), s * 0.8, (170, 170, 170), 1)
    y += int(8 * s / 0.55)
    row_h = int(34 * s / 0.55)
    for rank, g in enumerate(groups[:max_rows], 1):
        y += row_h
        f = g.features
        cells = [str(rank), g.level, f"{g.risk:.2f}", str(g.count), f"{f.get('severity', 0):.0%}",
                 f"{f.get('stranded_s', 0):.0f}s" if use_memory else "-", f"{f.get('isolation', 0):.2f}"]
        cv2.rectangle(panel, (14, y - int(16 * s / 0.55)), (24, y + 2), LEVEL_BGR[g.level], -1)
        for (_, fx), cell in zip(cols, cells):
            _text(panel, cell, (32 + fx * (pw - 40), y), s * 0.8, (235, 235, 235), 1)
    if len(groups) > max_rows:
        _text(panel, f"... {len(groups) - max_rows} more", (32, y + row_h), s * 0.8, (170, 170, 170), 1)
    y = h - int(20 * s / 0.55)
    for i, (name, c) in enumerate(LEVEL_BGR.items()):
        x = 14 + i * (pw - 14) // 4
        cv2.rectangle(panel, (x, y - int(14 * s / 0.55)), (x + 12, y), c, -1)
        _text(panel, name, (x + 18, y), s * 0.75, (220, 220, 220), 1)
    return np.hstack([img, panel])


In [ ]:
%%writefile /content/rescue/pipeline.py
"""
One frame through all five modules. Used by run_rescue.py and the evaluation scripts.

Model 2 is the slow part (a ResNet-50 U-Net at 1024x768), and flood water does
not change in a fraction of a second, so it runs every `severity_every_s`
seconds. Between runs the last map is reused: group positions are carried
into that older frame through the camera-motion transform, so a drifting
drone still reads the water at the right place.
"""

import numpy as np

from . import risk
from .detect import ANIMAL, PERSON, VEHICLE, Dets
from .memory import Memory, group_people
from .motion import CameraMotion
from .severity import SeverityMap
from .track import Tracker


class RescuePipeline:
    def __init__(self, detector, severity_model=None, fps=30.0, video=True,
                 use_memory=True, use_severity=True, use_motion=True,
                 severity_every_s=1.0, link_factor=3.0, radius_factor=4.0,
                 isolation_full=0.25, memory_kw=None):
        self.detector, self.sev_model = detector, severity_model
        self.fps, self.video = fps, video
        self.use_memory = use_memory and video          # one photo has no history
        self.use_severity = use_severity and severity_model is not None
        self.link_factor, self.radius_factor = link_factor, radius_factor
        self.isolation_full = isolation_full
        self.sev_every = max(1, int(round(severity_every_s * fps))) if video else 1
        self.tracker = Tracker(fps=fps) if video else None
        self.motion = CameraMotion(enabled=use_motion and video)
        self.memory = Memory(**(memory_kw or {}))
        self.smap, self.seg, self.seg_T = None, None, np.eye(3)
        self.i = -1

    # ------------------------------------------------------------------ helpers
    def _to_seg_frame(self, pts):
        """Current-frame points -> coordinates of the frame the severity map came from."""
        scene = self.motion.to_scene(pts)
        inv = np.linalg.inv(self.seg_T)
        return scene @ inv[:2, :2].T + inv[:2, 2]

    def _tracks(self, frame, dets: Dets):
        if self.tracker is not None:
            if not self.motion.enabled:
                return self.tracker.update(dets, frame)
            # Track in scene coordinates: ByteTrack matches by box overlap, and a
            # 15 px person shifts more than their own width when the drone pans.
            # Compensated, a still person stays put and the Kalman filter only
            # has to follow real movement. Boxes go back to frame coordinates after.
            z = self.motion.zoom
            c = self.motion.to_scene(dets.xywh[:, :2]) if len(dets) else np.zeros((0, 2))
            wh = dets.xywh[:, 2:] * z
            scene = Dets(np.hstack([c - wh / 2, c + wh / 2]), dets.conf, dets.cls)
            t = self.tracker.update(scene, frame)
            if len(t):
                inv = np.linalg.inv(self.motion.T)
                cc = (t[:, :2] + t[:, 2:4]) / 2
                cc = cc @ inv[:2, :2].T + inv[:2, 2]
                hw = (t[:, 2:4] - t[:, :2]) / (2 * z)
                t[:, :4] = np.hstack([cc - hw, cc + hw])
            return t
        ids = np.arange(1, len(dets) + 1, dtype=np.float32)
        return np.column_stack([dets.xyxy, ids, dets.conf, dets.cls]) if len(dets) else np.zeros((0, 7))

    # ------------------------------------------------------------------ main
    def step(self, frame):
        self.i += 1
        t = self.i / self.fps
        T = self.motion.update(frame)
        dets = self.detector(frame)
        tracks = self._tracks(frame, dets)
        persons = tracks[tracks[:, 6] == PERSON]
        others = tracks[np.isin(tracks[:, 6], [VEHICLE, ANIMAL])]

        fresh = False
        if self.use_severity and (self.smap is None or self.i % self.sev_every == 0):
            self.seg = self.sev_model.segment(frame)
            self.smap = SeverityMap(self.seg, self.isolation_full)
            self.seg_T, fresh = T.copy(), True

        if len(persons):
            b = persons[:, :4]
            centres = np.stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2], 1)
            sizes = np.maximum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]) * self.motion.zoom
            self.memory.observe_frame(t, persons[:, 4], self.motion.to_scene(centres), sizes)

        groups = group_people(persons, self.link_factor)
        for g in groups:
            sev = iso = 0.0
            if self.smap is not None:
                cx, cy = self._to_seg_frame(g.center)[0]
                radius = max(self.radius_factor * g.person_size, g.spread + g.person_size)
                sev, iso = self.smap.severity(cx, cy, radius), self.smap.isolation(cx, cy)
                if fresh:
                    for tid in g.members:
                        self.memory.observe_severity(tid, t, sev)
            stranded = float(np.median([self.memory.stranded_s(m, t) for m in g.members]))
            trend = float(np.mean([self.memory.trend_per_min(m, t) for m in g.members]))
            f = risk.factors(g.count, sev, stranded, iso, trend)
            g.risk = risk.score(f, self.use_memory, self.use_severity)
            g.level = risk.level(g.risk)
            g.features = dict(severity=sev, isolation=iso, stranded_s=stranded,
                              trend_per_min=trend, **{f"f_{k}": v for k, v in f.items()})
        groups.sort(key=lambda g: -g.risk)
        self.memory.forget(t)
        return dict(t=t, persons=persons, others=others, groups=groups, dets=dets,
                    seg=self.seg if self.use_severity else None)


In [ ]:
%%writefile /content/run_rescue.py
"""
Phases 3-5: run the full rescue pipeline on drone footage or photos.

    detect (Model 1 [+ SAHI]) -> track (ByteTrack) -> group -> memory
    -> flood severity (Model 2) -> risk score -> priority map / table / alerts

Inputs:
  * a video                    -> tracked over time; memory gives time stranded and water trend
  * a folder of frames, --sequence  -> same, frames in name order at --fps
  * a photo or folder of photos -> each photo on its own (no tracking, no memory)

Outputs in --out:
  annotated.mp4 / annotated/*.jpg   frames with boxes, groups, flood overlay and the priority panel
  priority_map.jpg                  the last (video) or each (photo) frame with the ranked table
  victims.csv                       one row per group: rank, people, position, every factor, risk, level
  tracks.csv                        (video) every person box per frame with its track and group
  alerts.json                       (video) when a group first reached High / Critical
  summary.json                      settings, counts, speed

Usage:
    python scripts/run_rescue.py --source flood.mp4 --out outputs/rescue/demo
    python scripts/run_rescue.py --source data/flood_c2a/test --out outputs/rescue/test_photos
    python scripts/run_rescue.py --source flood.mp4 --sahi --no-memory   # ablation
"""

import argparse
import csv
import json
import time
from pathlib import Path

import cv2
import numpy as np

from rescue.detect import Detector
from rescue.pipeline import RescuePipeline
from rescue.render import draw_frame, priority_panel
from rescue.severity import SeverityModel

ROOT = Path(__file__).resolve().parent.parent
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VID_EXT = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}
GROUP_FIELDS = ["rank", "group", "people", "cx", "cy", "level", "risk", "severity", "isolation",
                "stranded_s", "trend_per_min", "f_size", "f_severity", "f_stranded", "f_isolation",
                "f_trend", "members"]


def frames_of(src: Path, sequence: bool, max_frames: int):
    """Yields (name, frame). Video -> sequence; folder -> photos unless --sequence."""
    if src.suffix.lower() in VID_EXT:
        cap = cv2.VideoCapture(str(src))
        i = 0
        while i < max_frames:
            ok, f = cap.read()
            if not ok:
                break
            yield f"{i:06d}", f
            i += 1
        cap.release()
        return
    files = [src] if src.is_file() else sorted(p for p in src.iterdir() if p.suffix.lower() in IMG_EXT)
    for p in files[:max_frames]:
        f = cv2.imread(str(p))
        if f is not None:
            yield p.stem, f


def group_rows(groups, name=None):
    rows = []
    for rank, g in enumerate(groups, 1):
        f = g.features
        r = dict(rank=rank, group=f"G{g.gid}", people=g.count, cx=round(float(g.center[0]), 1),
                 cy=round(float(g.center[1]), 1), level=g.level, risk=round(g.risk, 3),
                 members=" ".join(map(str, g.members)),
                 **{k: round(float(f[k]), 3) for k in GROUP_FIELDS if k in f})
        if name is not None:
            r = {"image": name, **r}
        rows.append(r)
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", required=True, help="video, image, or folder")
    ap.add_argument("--out", required=True)
    ap.add_argument("--detector", default=str(ROOT / "models/flood_rehearsal.pt"))
    ap.add_argument("--severity", default=str(ROOT / "outputs/model2/runs/unet_r50_scse_hybrid/best.pt"))
    ap.add_argument("--imgsz", type=int, default=960)
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--sahi", action="store_true", help="tiled inference for small people")
    ap.add_argument("--slice", type=int, default=512)
    ap.add_argument("--sequence", action="store_true", help="treat an image folder as video frames")
    ap.add_argument("--fps", type=float, default=0, help="frame rate (default: the video's; 2 for frame folders)")
    ap.add_argument("--stride", type=int, default=1, help="process every n-th video frame")
    ap.add_argument("--max-frames", type=int, default=10**9)
    ap.add_argument("--severity-every", type=float, default=1.0, help="seconds between Model 2 runs")
    ap.add_argument("--no-memory", action="store_true")
    ap.add_argument("--no-severity", action="store_true")
    ap.add_argument("--no-motion", action="store_true", help="assume a fixed camera")
    ap.add_argument("--overlay", type=float, default=0.35, help="flood-map opacity (0 = off)")
    ap.add_argument("--no-panel", action="store_true")
    ap.add_argument("--device", default=None)
    args = ap.parse_args()

    src, out = Path(args.source), Path(args.out)
    if not src.exists():
        raise SystemExit(f"not found: {src}")
    out.mkdir(parents=True, exist_ok=True)
    video = src.suffix.lower() in VID_EXT or args.sequence

    fps = args.fps
    if src.suffix.lower() in VID_EXT:
        cap = cv2.VideoCapture(str(src))
        src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        n_src = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        fps = fps or src_fps / args.stride
        print(f"[video] {src.name}: {n_src} frames @ {src_fps:.1f} fps -> processing at {fps:.1f} fps")
    fps = fps or 2.0

    print(f"[model 1] {args.detector}" + ("  + SAHI" if args.sahi else ""))
    det = Detector(args.detector, imgsz=args.imgsz, conf=args.conf, device=args.device,
                   sahi=args.sahi, slice_size=args.slice)
    sev = None
    if not args.no_severity:
        sev = SeverityModel(args.severity, device=args.device)
        print(f"[model 2] {args.severity} ({sev.run}, {sev.device})")

    def new_pipeline():
        return RescuePipeline(det, sev, fps=fps, video=video, use_memory=not args.no_memory,
                              use_severity=not args.no_severity, use_motion=not args.no_motion,
                              severity_every_s=args.severity_every)

    pipe = new_pipeline()
    writer, last_img, last_groups = None, None, []
    group_csv, track_rows, alerts, best_level = [], [], [], {}
    rank_of_level = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
    t0, n = time.time(), 0
    if not video:
        (out / "annotated").mkdir(exist_ok=True)

    raw = frames_of(src, args.sequence, args.max_frames * args.stride)
    for k, (name, frame) in enumerate(raw):
        if src.suffix.lower() in VID_EXT and k % args.stride:
            continue
        if not video:
            pipe = new_pipeline()                          # photos are independent
        r = pipe.step(frame)
        n += 1
        img = draw_frame(frame, r["persons"], r["others"], r["groups"], r["seg"],
                         overlay_alpha=args.overlay, t=r["t"] if video else None)
        if not args.no_panel:
            img = priority_panel(img, r["groups"], use_memory=pipe.use_memory)
        last_img, last_groups = img, r["groups"]

        if video:
            if writer is None:
                h, w = img.shape[:2]
                writer = cv2.VideoWriter(str(out / "annotated.mp4"), cv2.VideoWriter_fourcc(*"mp4v"),
                                         fps, (w, h))
            writer.write(img)
            gid_of = {m: g.gid for g in r["groups"] for m in g.members}
            for x1, y1, x2, y2, tid, sc, cl in np.vstack([r["persons"], r["others"]]):
                track_rows.append(dict(frame=pipe.i, t=round(r["t"], 2), track=int(tid), cls=int(cl),
                                       score=round(float(sc), 3), x1=round(float(x1), 1), y1=round(float(y1), 1),
                                       x2=round(float(x2), 1), y2=round(float(y2), 1),
                                       group=f"G{gid_of[int(tid)]}" if int(tid) in gid_of else ""))
            for g in r["groups"]:
                if rank_of_level[g.level] >= 2 and rank_of_level[g.level] > best_level.get(g.gid, -1):
                    alerts.append(dict(t=round(r["t"], 2), frame=pipe.i, group=f"G{g.gid}", level=g.level,
                                       risk=round(g.risk, 3), people=g.count,
                                       position=[round(float(v), 1) for v in g.center],
                                       severity=round(g.features["severity"], 3),
                                       stranded_s=round(g.features["stranded_s"], 1)))
                    print(f"  ALERT t={r['t']:.1f}s  G{g.gid} -> {g.level} (risk {g.risk:.2f}, {g.count} people)")
                best_level[g.gid] = max(best_level.get(g.gid, -1), rank_of_level[g.level])
            if n % 50 == 0:
                print(f"  frame {n}  {n / (time.time() - t0):.2f} fps  people {len(r['persons'])}  "
                      f"groups {len(r['groups'])}")
        else:
            cv2.imwrite(str(out / "annotated" / f"{name}.jpg"), img)
            group_csv += group_rows(r["groups"], name)
            print(f"  {name[:40]:40s} people {len(r['persons']):3d}  groups {len(r['groups']):2d}  "
                  f"top {r['groups'][0].level if r['groups'] else '-'}")

    if n == 0:
        raise SystemExit("no frames read")
    elapsed = time.time() - t0
    if writer is not None:
        writer.release()
    cv2.imwrite(str(out / "priority_map.jpg"), last_img)

    if video:
        group_csv = group_rows(last_groups)
        _write_csv(out / "tracks.csv", track_rows)
        (out / "alerts.json").write_text(json.dumps(alerts, indent=2))
    _write_csv(out / "victims.csv", group_csv)

    summary = dict(source=str(src), mode="video" if video else "photos", frames=n,
                   seconds=round(elapsed, 1), fps_processed=round(n / elapsed, 2),
                   detector=args.detector, sahi=args.sahi, severity=None if args.no_severity else args.severity,
                   use_memory=pipe.use_memory, use_severity=pipe.use_severity,
                   camera_motion=not args.no_motion and video,
                   motion_failures=pipe.motion.failures if video else None,
                   alerts=len(alerts), final_groups=len(last_groups),
                   final_levels={lv: sum(g.level == lv for g in last_groups) for lv in rank_of_level})
    if video:
        summary["unique_tracks"] = len({row["track"] for row in track_rows if row["cls"] == 0})
    (out / "summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    print(f"[done] {out}")


def _write_csv(path, rows):
    if not rows:
        path.write_text("")
        return
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/eval_sahi.py
"""
Phase 3 comparison: plain YOLO vs YOLO + SAHI, on the 109 real flood test photos.

Same detector (models/flood_rehearsal.pt), same images, same confidence
floor, same scorer - only tiling changes. Scored with pycocotools (COCO
protocol), which also gives the small-object AP the project plan asks for
(COCO "small" = under 32x32 px). Every Roboflow category in the export is a
person, so all ground truth is merged into one "person" class.

pycocotools interpolates precision slightly differently from Ultralytics,
so the plain-YOLO row here will not match train_flood.py's 0.399 to the
third decimal. Compare rows within this table, not across tools.

SAHI is built for high-resolution footage where people are a few pixels
tall. The flood test photos are only 640 px, so the same comparison should
also be run on VisDrone val (1-2K px, many tiny people) via --yolo-labels.

Usage:
    python scripts/eval_sahi.py --images data/flood_c2a/test \\
        --coco data/flood_c2a/test/_annotations.coco.json --out outputs/rescue/sahi_eval.json
    python scripts/eval_sahi.py ... --slices 256 320 416 --max-images 20
    python scripts/eval_sahi.py --images /content/datasets/VisDrone/VisDrone2019-DET-val/images \
        --yolo-labels /content/datasets/VisDrone/VisDrone2019-DET-val/labels --slices 512 640 \
        --out outputs/rescue/sahi_eval_visdrone.json
"""

import argparse
import contextlib
import copy
import io
import json
import time
from pathlib import Path

import cv2
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from rescue.detect import PERSON, Detector

ROOT = Path(__file__).resolve().parent.parent
STAT_NAMES = ["AP50-95", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
              "AR@10", "AR@100", "AR@300", "AR_small", "AR_medium", "AR_large"]


def person_only(gt_json):
    d = copy.deepcopy(gt_json)
    d["categories"] = [{"id": 1, "name": "person", "supercategory": "none"}]
    for a in d["annotations"]:
        a["category_id"] = 1
    return d


def yolo_to_coco(images_dir, labels_dir, cls=0):
    """COCO ground truth from YOLO txt labels, keeping one class (person = 0)."""
    from PIL import Image
    images, anns = [], []
    files = sorted(p for p in Path(images_dir).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    for iid, p in enumerate(files):
        w, h = Image.open(p).size
        images.append(dict(id=iid, file_name=p.name, width=w, height=h))
        lp = Path(labels_dir) / f"{p.stem}.txt"
        for line in (lp.read_text().splitlines() if lp.exists() else []):
            v = line.split()
            if len(v) == 5 and int(v[0]) == cls:
                cx, cy, bw, bh = (float(x) for x in v[1:])
                anns.append(dict(id=len(anns) + 1, image_id=iid, category_id=1, iscrowd=0,
                                 bbox=[(cx - bw / 2) * w, (cy - bh / 2) * h, bw * w, bh * h],
                                 area=bw * w * bh * h))
    return dict(images=images, annotations=anns,
                categories=[{"id": 1, "name": "person", "supercategory": "none"}])


def evaluate(gt, dets, img_ids):
    if not dets:
        return {k: 0.0 for k in STAT_NAMES}
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(dets)
        E = COCOeval(gt, dt, "bbox")
        E.params.imgIds = img_ids
        E.params.maxDets = [10, 100, 300]          # crowded scenes: up to ~300 people
        E.evaluate()
        E.accumulate()
        E.summarize()
    return {k: round(float(v), 4) for k, v in zip(STAT_NAMES, E.stats)}


def run(detector, gt, images_dir, img_ids):
    dets, secs, n_boxes = [], 0.0, 0
    for iid in img_ids:
        info = gt.imgs[iid]
        frame = cv2.imread(str(images_dir / info["file_name"]))
        if frame is None:
            raise SystemExit(f"missing image {info['file_name']}")
        t = time.perf_counter()
        d = detector(frame)
        secs += time.perf_counter() - t
        keep = d.cls == PERSON
        for (x1, y1, x2, y2), s in zip(d.xyxy[keep], d.conf[keep]):
            dets.append(dict(image_id=iid, category_id=1, score=float(s),
                             bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)]))
        n_boxes += int(keep.sum())
    return dets, secs / len(img_ids), n_boxes


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--images", default=str(ROOT / "data/flood_c2a/test"))
    ap.add_argument("--coco", default=str(ROOT / "data/flood_c2a/test/_annotations.coco.json"))
    ap.add_argument("--yolo-labels", default=None, help="YOLO txt label dir instead of --coco (person = 0)")
    ap.add_argument("--weights", default=str(ROOT / "models/flood_rehearsal.pt"))
    ap.add_argument("--imgsz", type=int, default=960)
    ap.add_argument("--conf", type=float, default=0.01, help="low floor, as for any mAP evaluation")
    ap.add_argument("--slices", type=int, nargs="+", default=[320, 416])
    ap.add_argument("--overlap", type=float, default=0.2)
    ap.add_argument("--max-images", type=int, default=0)
    ap.add_argument("--device", default=None)
    ap.add_argument("--out", default=str(ROOT / "outputs/rescue/sahi_eval.json"))
    args = ap.parse_args()

    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO()
        gt.dataset = (yolo_to_coco(args.images, args.yolo_labels) if args.yolo_labels
                      else person_only(json.loads(Path(args.coco).read_text())))
        gt.createIndex()
    img_ids = sorted(gt.getImgIds())[: args.max_images or None]
    n_gt = sum(len(gt.getAnnIds(imgIds=[i])) for i in img_ids)
    print(f"[data] {len(img_ids)} images, {n_gt} person boxes")

    configs = [("YOLO", dict(sahi=False))] + [
        (f"YOLO + SAHI ({s}px tiles)", dict(sahi=True, slice_size=s)) for s in args.slices]
    rows = []
    for name, kw in configs:
        det = Detector(args.weights, imgsz=args.imgsz, conf=args.conf, device=args.device,
                       overlap=args.overlap, **kw)
        dets, sec, nb = run(det, gt, Path(args.images), img_ids)
        m = evaluate(gt, dets, img_ids)
        rows.append(dict(method=name, sec_per_image=round(sec, 3), boxes=nb, **m))
        print(f"{name:28s} AP50 {m['AP50']:.3f}  AP50-95 {m['AP50-95']:.3f}  "
              f"AP_small {m['AP_small']:.3f}  AR@300 {m['AR@300']:.3f}  {sec:.2f}s/img")

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    res = dict(weights=args.weights, imgsz=args.imgsz, conf=args.conf, overlap=args.overlap,
               images=len(img_ids), gt_boxes=n_gt, scorer="pycocotools (COCO protocol), person only",
               results=rows)
    out.write_text(json.dumps(res, indent=2))

    keys = ["AP50", "AP50-95", "AP75", "AP_small", "AP_medium", "AR@300", "sec_per_image"]
    md = ["| Method | " + " | ".join(keys) + " |", "|---" * (len(keys) + 1) + "|"]
    for r in rows:
        md.append(f"| {r['method']} | " + " | ".join(f"{r[k]:.3f}" if k != "sec_per_image"
                                                    else f"{r[k]:.2f}" for k in keys) + " |")
    md.append(f"\n{len(img_ids)} images from {Path(args.images).name} ({n_gt} people), COCO protocol via pycocotools, "
              f"conf >= {args.conf}, imgsz {args.imgsz}. AP_small = people under 32x32 px. "
              f"sec_per_image depends on the GPU.")
    out.with_suffix(".md").write_text("\n".join(md) + "\n")
    print("\n".join(md))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/eval_priority.py
"""
Phase 5 evaluation: does the risk score rank groups the way people would?

There is no public benchmark of "who should be rescued first", so this uses
human judgement, the standard fallback:

  1. `prepare`: runs Models 1 + 2 on photos that contain 2+ groups and draws
     every group with a number, in RANDOM order (no risk shown, so the
     raters are not anchored). Writes sheet/<photo>.jpg and labels.csv.
  2. Each rater opens the sheets and fills in their column of labels.csv
     with the group number they would send the boat to FIRST (0 = no box
     contains real people; see priority_eval/README_RATERS.md).
  3. `score`: compares every ranking variant with the raters' choices:
       full          all factors (single photos have no memory)
       no_severity   people count only (Model 2 switched off)
       severity_only flood factors only (severity + isolation)
       random        expected top-1 for a random pick = mean(1 / groups)
     Reports top-1 agreement with the majority vote, and how often the raters
     agree with each other (the ceiling any method can reach).

Memory (time stranded, water trend) needs video and is not covered here;
its effect is shown on the demo video instead.

Usage:
    python scripts/eval_priority.py prepare --images data/flood_c2a/test --out outputs/rescue/priority_eval
    # raters fill labels.csv columns rater1, rater2, rater3
    python scripts/eval_priority.py score      # plain Python + numpy + opencv is enough
"""

import argparse
import csv
import json
import random
from collections import Counter
from pathlib import Path

import cv2
import numpy as np

from rescue import risk          # models are imported inside prepare(), so `score` runs anywhere

ROOT = Path(__file__).resolve().parent.parent
VARIANTS = {
    "full": dict(use_memory=False, use_severity=True),
    "no_severity": dict(use_memory=False, use_severity=False),
    "severity_only": dict(weights=dict(severity=0.75, isolation=0.25)),
}


def prepare(args):
    from rescue.detect import Detector
    from rescue.pipeline import RescuePipeline
    from rescue.render import _text
    from rescue.severity import SeverityModel
    det = Detector(args.detector, imgsz=args.imgsz, conf=args.conf, device=args.device)
    sev = SeverityModel(args.severity, device=args.device)
    out = Path(args.out)
    (out / "sheet").mkdir(parents=True, exist_ok=True)
    rng = random.Random(0)
    files = sorted(p for p in Path(args.images).iterdir() if p.suffix.lower() in {".jpg", ".png", ".jpeg"})
    rows, groups_json = [], {}
    for p in files:
        frame = cv2.imread(str(p))
        r = RescuePipeline(det, sev, video=False).step(frame)
        gs = [g for g in r["groups"]]
        if len(gs) < 2:
            continue
        rng.shuffle(gs)                                   # numbering carries no information
        img = frame.copy()
        s = max(0.6, frame.shape[1] / 1100)
        for k, g in enumerate(gs, 1):
            pad = 0.5 * g.person_size
            x1, y1, x2, y2 = (g.bbox + np.array([-pad, -pad, pad, pad])).astype(int)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), 2)
            _text(img, str(k), (x1 + 2, max(18, y1 - 4)), s, (0, 0, 0), 2, bg=(255, 255, 255))
        cv2.imwrite(str(out / "sheet" / f"{p.stem}.jpg"), img)
        groups_json[p.stem] = [dict(number=k, people=g.count, **{f: g.features[f] for f in
                                    ("severity", "isolation")}) for k, g in enumerate(gs, 1)]
        rows.append(dict(photo=p.stem, groups=len(gs), rater1="", rater2="", rater3=""))
        if len(rows) >= args.max_photos:
            break
    (out / "groups.json").write_text(json.dumps(groups_json, indent=1))
    with open(out / "labels.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    print(f"[prepare] {len(rows)} photos with 2+ groups -> {out / 'sheet'}")
    print(f"Raters: open each sheet, write the number of the group to rescue FIRST in labels.csv.")


def rank_top(groups, variant):
    v = VARIANTS[variant]
    def sc(g):
        f = risk.factors(g["people"], g["severity"], 0, g["isolation"], 0)
        if "weights" in v:
            return risk.score(f, weights=v["weights"])
        return risk.score(f, v["use_memory"], v["use_severity"])
    best = max(sc(g) for g in groups)
    return {g["number"] for g in groups if abs(sc(g) - best) < 1e-9}       # ties count as a hit


def score(args):
    out = Path(args.out)
    groups = json.loads((out / "groups.json").read_text())
    rows = list(csv.DictReader(open(out / "labels.csv")))
    raters = [c for c in rows[0] if c.startswith("rater")]
    used, hits, chance, pair_agree, pairs, no_people = 0, Counter(), 0.0, 0, 0, 0
    for r in rows:
        votes = [int(r[c]) for c in raters if r[c].strip()]
        if not votes:
            continue
        top, n = Counter(votes).most_common(1)[0]
        if list(Counter(votes).values()).count(n) > 1:
            continue                                           # no majority -> skip the photo
        if top == 0:
            no_people += 1                                     # majority: no real people in any box
            continue
        used += 1
        for v in VARIANTS:
            top_set = rank_top(groups[r["photo"]], v)
            hits[v] += (top in top_set) / len(top_set)        # a tie of k groups earns 1/k
        chance += 1 / len(groups[r["photo"]])
        for i in range(len(votes)):
            for j in range(i + 1, len(votes)):
                pairs += 1
                pair_agree += votes[i] == votes[j]
    if not used:
        raise SystemExit("no labelled photos with a majority vote yet")
    res = dict(photos=used, skipped_no_real_people=no_people, raters=raters,
               top1_vs_majority={v: round(hits[v] / used, 3) for v in VARIANTS},
               random_top1=round(chance / used, 3),
               rater_pairwise_agreement=round(pair_agree / pairs, 3) if pairs else None)
    (out / "priority_eval.json").write_text(json.dumps(res, indent=2))
    print(json.dumps(res, indent=2))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["prepare", "score"])
    ap.add_argument("--images", default=str(ROOT / "data/flood_c2a/test"))
    ap.add_argument("--out", default=str(ROOT / "outputs/rescue/priority_eval"))
    ap.add_argument("--detector", default=str(ROOT / "models/flood_rehearsal.pt"))
    ap.add_argument("--severity", default=str(ROOT / "outputs/model2/runs/unet_r50_scse_hybrid/best.pt"))
    ap.add_argument("--imgsz", type=int, default=960)
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--max-photos", type=int, default=40)
    ap.add_argument("--device", default=None)
    args = ap.parse_args()
    prepare(args) if args.cmd == "prepare" else score(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/make_demo_video.py
"""
Builds a SIMULATED drone pass over one still aerial photo, for testing the
tracker, camera-motion compensation and memory when no real flood video is
at hand.

A crop window drifts across the photo with slow zoom and slight rotation,
the way a hovering drone wanders. What it can test: IDs stay stable while
the camera moves, still people are recognised as still (time stranded
grows), the flood map stays aligned. What it CANNOT show: people moving or
water rising - nothing in a still photo changes. Label any result from it
as "simulated camera motion over a still image".

Usage:
    python scripts/make_demo_video.py --image some_flood_photo.jpg --out demo.mp4 --seconds 30 --fps 10
"""

import argparse
import math

import cv2
import numpy as np


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--seconds", type=float, default=30)
    ap.add_argument("--fps", type=float, default=10)
    ap.add_argument("--size", type=int, nargs=2, default=[1280, 720], help="output W H")
    ap.add_argument("--view", type=float, default=0.75, help="crop width as a share of the photo")
    ap.add_argument("--rotate", type=float, default=3.0, help="max rotation, degrees")
    args = ap.parse_args()

    img = cv2.imread(args.image)
    if img is None:
        raise SystemExit(f"cannot read {args.image}")
    H, W = img.shape[:2]
    ow, oh = args.size
    n = int(args.seconds * args.fps)
    vw = cv2.VideoWriter(args.out, cv2.VideoWriter_fourcc(*"mp4v"), args.fps, (ow, oh))
    for i in range(n):
        u = i / max(1, n - 1)
        zoom = 1.0 + 0.15 * math.sin(math.pi * u)                    # descend a little, then back
        cw = args.view * W / zoom
        ch = cw * oh / ow
        if ch > H * 0.95:
            ch = H * 0.95
            cw = ch * ow / oh
        # centre wanders on a slow ellipse inside the free margin
        mx, my = (W - cw) / 2, (H - ch) / 2
        cx = W / 2 + 0.8 * mx * math.sin(2 * math.pi * u)
        cy = H / 2 + 0.8 * my * math.sin(math.pi * u)
        ang = args.rotate * math.sin(2 * math.pi * u * 1.5)
        s = ow / cw
        M = cv2.getRotationMatrix2D((cx, cy), ang, s)
        M[:, 2] += np.array([ow / 2 - cx, oh / 2 - cy])
        vw.write(cv2.warpAffine(img, M, (ow, oh), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT))
    vw.release()
    print(f"[demo] {n} frames @ {args.fps} fps -> {args.out}  (simulated camera motion over a still photo)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/_test_rescue.py
"""
Tests for the rescue pipeline's logic modules (no GPU, no weights needed):
grouping, memory (stranded time, water trend), risk formula and ablations,
severity lookups, camera-motion compensation.

    python scripts/_test_rescue.py
"""

import numpy as np

from rescue import risk
from rescue.memory import Memory, group_people
from rescue.motion import CameraMotion
from rescue.severity import CLASSES, SeverityMap


def box(cx, cy, s, tid):
    return [cx - s / 2, cy - s / 2, cx + s / 2, cy + s / 2, tid, 0.9, 0]


def test_grouping():
    t = np.array([box(100, 100, 20, 1), box(130, 100, 20, 2), box(160, 110, 20, 3),   # chain -> 1 group
                  box(600, 600, 20, 4)], np.float32)                                    # alone
    gs = sorted(group_people(t, link_factor=3.0), key=lambda g: -g.count)
    assert [g.count for g in gs] == [3, 1], [g.count for g in gs]
    assert gs[0].gid == 1 and set(gs[0].members) == {1, 2, 3}
    assert group_people(np.zeros((0, 7)), 3.0) == []
    crowd = np.array([box(100 + 30 * k, 100, 20, k + 1) for k in range(40)], np.float32)   # 1200 px line
    gs = group_people(crowd, link_factor=3.0, max_extent=12.0)
    assert sum(g.count for g in gs) == 40 and len(gs) > 1
    assert all(g.boxes[:, 2].max() - g.boxes[:, 0].min() <= 12 * 20 + 20 for g in gs)


def test_memory_stranded_and_trend():
    m = Memory(move_factor=1.0, trend_window_s=60, trend_min_s=10, ttl_s=5)
    for k in range(61):                       # still for 60 s, water rising 0.2 -> 0.5
        t = float(k)
        m.observe(7, t, (100 + 0.2 * np.sin(k), 100), 20)
        m.observe_severity(7, t, 0.2 + 0.005 * k)
    assert abs(m.stranded_s(7, 60) - 60) < 1e-6
    assert abs(m.trend_per_min(7, 60) - 0.3) < 1e-6          # 0.005/s = 0.3/min
    m.observe(7, 61, (200, 100), 20)                         # walks 5 body sizes
    assert m.stranded_s(7, 61) == 0
    m.forget(100)
    assert 7 not in m.tracks
    # re-identification by place: track 7 lost, track 9 appears on the same spot
    m = Memory(move_factor=1.0, ttl_s=10)
    for k in range(20):
        m.observe_frame(float(k), [7, 8], [(100, 100), (400, 100)], [20, 20])
    m.observe_frame(21.0, [9, 8], [(103, 98), (400, 100)], [20, 20])
    assert 7 not in m.tracks and abs(m.stranded_s(9, 21) - 21) < 1e-6
    m.observe_frame(22.0, [10, 8, 9], [(400, 104), (400, 100), (103, 98)], [20, 20, 20])
    assert m.stranded_s(10, 22) == 0                         # 8 is present -> 10 is a new person
    m2 = Memory()
    m2.observe(1, 0, (0, 0), 10)
    m2.observe_severity(1, 0, 0.5)
    assert m2.trend_per_min(1, 0) == 0.0                     # too little history -> no trend


def test_risk():
    hi = risk.factors(count=12, severity=0.9, stranded_s=600, isolation=1.0, trend_per_min=0.2)
    lo = risk.factors(count=1, severity=0.0, stranded_s=0, isolation=0.0, trend_per_min=-0.1)
    assert risk.score(hi) > 0.95 and risk.level(risk.score(hi)) == "Critical"
    assert risk.score(lo) < 0.05 and risk.level(risk.score(lo)) == "Low"
    assert abs(sum(risk.WEIGHTS.values()) - 1) < 1e-9
    f = risk.factors(count=5, severity=0.8, stranded_s=0, isolation=0.5, trend_per_min=0)
    # no memory: stranded/trend dropped and weights renormalised
    w = {k: v for k, v in risk.WEIGHTS.items() if k not in risk.MEMORY_FACTORS}
    expect = sum(w[k] * f[k] for k in w) / sum(w.values())
    assert abs(risk.score(f, use_memory=False) - expect) < 1e-9
    # no severity: only size + stranded remain
    assert abs(risk.score(f, use_severity=False) - (0.2 * 0.5) / (0.2 + 0.25)) < 1e-9
    assert [risk.level(x) for x in (0.81, 0.8, 0.5, 0.31, 0.3, 0.0)] == \
        ["Critical", "High", "Medium", "Medium", "Low", "Low"]


def test_severity_map():
    W = CLASSES.index("water")
    seg = np.full((200, 200), CLASSES.index("grass"), np.uint8)   # dry mainland on the left
    seg[:, 100:] = W
    seg[90:110, 160:180] = CLASSES.index("building-non-flooded")  # dry island (roof) in the water
    s = SeverityMap(seg, isolation_full=0.25)
    assert s.severity(185, 30, 10) == 1.0
    assert s.severity(20, 100, 15) == 0.0
    assert 0.4 < s.severity(100, 30, 30) < 0.6
    assert s.isolation(20, 100) == 0.0                                 # on mainland
    assert s.isolation(170, 100) > 0.8                                 # on the island: ~70 px of water
    assert s.isolation(120, 100) < s.isolation(170, 100)
    no_road = SeverityMap(np.full((50, 50), CLASSES.index("water"), np.uint8))
    assert no_road.isolation(10, 10) == 1.0
    bg = SeverityMap(np.zeros((50, 50), np.uint8))           # all background -> no evidence
    assert bg.severity(25, 25, 10) == 0.0


def test_camera_motion():
    rng = np.random.default_rng(0)
    base = (rng.random((480, 640)) * 255).astype(np.uint8)
    base = np.dstack([cv2_blur(base)] * 3)
    cm = CameraMotion(work_width=640)
    cm.update(base)
    M = np.float32([[1, 0, -12], [0, 1, 7]])                 # camera moved: scene shifted by (-12, +7)
    import cv2
    moved = cv2.warpAffine(base, M, (640, 480), borderMode=cv2.BORDER_REFLECT)
    cm.update(moved)
    p = cm.to_scene([[300, 200]])[0]                         # pixel in moved frame -> first frame
    assert np.allclose(p, [312, 193], atol=0.6), p


def cv2_blur(x):
    import cv2
    return cv2.GaussianBlur(x, (0, 0), 2)


if __name__ == "__main__":
    for name, fn in list(globals().items()):
        if name.startswith("test_"):
            fn()
            print(f"ok  {name}")
    print("all rescue tests passed")


In [ ]:
%%writefile /content/visdrone_to_yolo.py
"""
Phase 1 - Step 1: Convert VisDrone2019-DET annotations to YOLO format.

VisDrone annotation line:
    bbox_left,bbox_top,bbox_width,bbox_height,score,category,truncation,occlusion

VisDrone categories:
    0 ignored-regions  1 pedestrian  2 people   3 bicycle  4 car   5 van
    6 truck            7 tricycle    8 awning-tricycle     9 bus  10 motor
    11 others

Project class scheme (kept IDENTICAL for the Phase-2 flood fine-tune so the
YOLO detection head is never reinitialized):
    0 person   <- pedestrian, people
    1 vehicle  <- bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor
    2 animal   <- (none in VisDrone; appears only in the flood dataset)

No images are copied or moved. Labels are written to <split>/labels/ next to
<split>/images/, which is exactly where Ultralytics looks for them.

Usage:
    python visdrone_to_yolo.py --root /content/datasets/VisDrone
"""

import argparse
import json
from collections import Counter
from pathlib import Path

from PIL import Image

VISDRONE_TO_OURS = {
    1: 0, 2: 0,                                  # person
    3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1,  # vehicle
}
CLASS_NAMES = ["person", "vehicle", "animal"]
SPLITS = {"train": "VisDrone2019-DET-train", "val": "VisDrone2019-DET-val"}


def convert_split(split_dir: Path, min_side_px: float = 2.0) -> dict:
    img_dir, ann_dir = split_dir / "images", split_dir / "annotations"
    lbl_dir = split_dir / "labels"
    if not img_dir.is_dir() or not ann_dir.is_dir():
        raise FileNotFoundError(f"expected images/ and annotations/ inside {split_dir}")
    lbl_dir.mkdir(exist_ok=True)

    stats = {
        "images": 0, "images_with_no_objects": 0, "boxes_kept": 0,
        "skipped_ignored_region": 0, "skipped_other_class": 0,
        "skipped_degenerate": 0, "per_class": Counter(),
        # COCO definition of a "small" object: area < 32*32 px
        "small_boxes_coco": 0, "medium_boxes_coco": 0, "large_boxes_coco": 0,
    }

    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        ann_path = ann_dir / f"{img_path.stem}.txt"
        stats["images"] += 1
        with Image.open(img_path) as im:
            W, H = im.size

        lines = []
        if ann_path.exists():
            for raw in ann_path.read_text().splitlines():
                raw = raw.strip().rstrip(",")
                if not raw:
                    continue
                p = raw.split(",")
                if len(p) < 6:
                    continue
                x, y, w, h = (float(v) for v in p[:4])
                score, cat = int(float(p[4])), int(float(p[5]))

                if score == 0 or cat == 0:          # ignored region
                    stats["skipped_ignored_region"] += 1
                    continue
                if cat not in VISDRONE_TO_OURS:     # category 11 "others"
                    stats["skipped_other_class"] += 1
                    continue

                # clamp to image bounds, then drop slivers
                x1, y1 = max(0.0, x), max(0.0, y)
                x2, y2 = min(float(W), x + w), min(float(H), y + h)
                bw, bh = x2 - x1, y2 - y1
                if bw < min_side_px or bh < min_side_px:
                    stats["skipped_degenerate"] += 1
                    continue

                cls = VISDRONE_TO_OURS[cat]
                lines.append(
                    f"{cls} {(x1 + bw / 2) / W:.6f} {(y1 + bh / 2) / H:.6f} "
                    f"{bw / W:.6f} {bh / H:.6f}"
                )
                stats["boxes_kept"] += 1
                stats["per_class"][CLASS_NAMES[cls]] += 1
                area = bw * bh
                if area < 32 ** 2:
                    stats["small_boxes_coco"] += 1
                elif area < 96 ** 2:
                    stats["medium_boxes_coco"] += 1
                else:
                    stats["large_boxes_coco"] += 1

        (lbl_dir / f"{img_path.stem}.txt").write_text("\n".join(lines))
        if not lines:
            stats["images_with_no_objects"] += 1

    stats["per_class"] = dict(stats["per_class"])
    return stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True,
                    help="dir containing VisDrone2019-DET-train / -val")
    ap.add_argument("--yaml-out", default=None,
                    help="path for the generated data.yaml (default <root>/visdrone.yaml)")
    ap.add_argument("--stats-out", default=None,
                    help="path for the conversion stats JSON")
    args = ap.parse_args()

    root = Path(args.root).resolve()
    all_stats = {}
    for split, folder in SPLITS.items():
        print(f"[convert] {split}: {folder}")
        all_stats[split] = convert_split(root / folder)
        s = all_stats[split]
        print(f"  images={s['images']}  boxes={s['boxes_kept']}  "
              f"per_class={s['per_class']}  "
              f"small(COCO)={s['small_boxes_coco']} "
              f"({100 * s['small_boxes_coco'] / max(1, s['boxes_kept']):.1f}%)  "
              f"empty_images={s['images_with_no_objects']}")

    yaml_path = Path(args.yaml_out) if args.yaml_out else root / "visdrone.yaml"
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    yaml_path.write_text(
        "# Auto-generated by visdrone_to_yolo.py - Phase 1 base detector\n"
        f"path: {root.as_posix()}\n"
        f"train: {SPLITS['train']}/images\n"
        f"val: {SPLITS['val']}/images\n"
        "nc: 3\n"
        "names:\n"
        "  0: person\n"
        "  1: vehicle\n"
        "  2: animal   # unused in VisDrone; reserved for the Phase-2 flood fine-tune\n"
    )
    print(f"[convert] wrote {yaml_path}")

    stats_path = Path(args.stats_out) if args.stats_out else root / "visdrone_conversion_stats.json"
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    stats_path.write_text(json.dumps(all_stats, indent=2))
    print(f"[convert] wrote {stats_path}")


if __name__ == "__main__":
    main()


In [ ]:
%cd /content
!python _test_rescue.py

## 5. Flood test photos (used by sections 7 and 10)

In [ ]:
SRC = '/content/datasets/flood_c2a_coco'
if not os.path.exists(f'{SRC}/test/_annotations.coco.json'):
    os.makedirs(SRC, exist_ok=True)
    !unzip -q "{C2A_ZIP}" "test/*" -d {SRC}
print(len(glob.glob(f'{SRC}/test/*.jpg')), 'test photos')

## 6. (OPTIONAL - skip it) SAHI on VisDrone val

**SAHI is dropped from the system**: on the 109 flood test photos it lowered AP50 from 0.404
to 0.326 (`outputs/rescue/sahi_eval_flood_cpu.md`). This only checks whether it would help
on high-resolution images (VisDrone val, 1-2K px). Nice for the discussion, not required.

In [ ]:
VD = '/content/datasets/VisDrone'
if not os.path.isdir(f'{VD}/VisDrone2019-DET-val'):
    os.makedirs(VD, exist_ok=True)
    !curl -sL -o /content/vdval.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip
    !unzip -q /content/vdval.zip -d {VD} && rm /content/vdval.zip
if not os.path.isdir(f'{VD}/VisDrone2019-DET-val/labels'):
    import sys; from pathlib import Path; sys.path.insert(0, '/content')
    from visdrone_to_yolo import convert_split            # same converter as Phase 1, val only
    print(convert_split(Path(f'{VD}/VisDrone2019-DET-val'))['per_class'])
print(len(glob.glob(f'{VD}/VisDrone2019-DET-val/labels/*.txt')), 'label files')
!python eval_sahi.py --images {VD}/VisDrone2019-DET-val/images \
    --yolo-labels {VD}/VisDrone2019-DET-val/labels \
    --weights {DET} --slices 512 640 --out {OUT}/sahi_eval_visdrone.json

## 7. Single photos - every flood test photo through the pipeline

No tracking or memory (one photo has no history). Writes `annotated/`, `victims.csv`.

In [ ]:
!python run_rescue.py --source {SRC}/test --out {OUT}/test_photos \
    --detector {DET} --severity {SEV} | tail -25
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob(f'{OUT}/test_photos/annotated/*.jpg'))[:4]:
    display(IPImage(p, width=900))

## 8. Video - the full system on EVERY clip in `videos/`

**Upload real aerial flood footage** to `MyDrive/FloodRescue/videos/` first - see
`docs/FINAL_DEMO_PLAN.md` for what makes a good clip. Top-down views work best (Model 2 was
trained on top-down FloodNet imagery).

No video yet? Falls back to a **simulated** drone pass over a test photo - label anything
from it "simulated camera motion over a still image" (nothing moves in it).

In [ ]:
vids = sorted(glob.glob(f'{VIDEOS}/*.mp4') + glob.glob(f'{VIDEOS}/*.mov') + glob.glob(f'{VIDEOS}/*.MP4'))
if not vids:
    STILL = glob.glob(f'{SRC}/test/400a88b7*.jpg')[0]      # flooded plaza, ~50 people
    !python make_demo_video.py --image "{STILL}"         --out /content/simulated_pass.mp4 --seconds 30 --fps 10 --size 640 360 --view 0.95 --rotate 2
    vids = ['/content/simulated_pass.mp4']
NAMES = []
for VIDEO in vids:
    NAME = os.path.splitext(os.path.basename(VIDEO))[0]; NAMES.append(NAME)
    print('=' * 20, NAME)
    # --stride 3: 30 fps footage processed at 10 fps - plenty for people who barely move
    !python run_rescue.py --source "{VIDEO}" --out "{OUT}/{NAME}/full" --detector {DET} --severity {SEV}         --stride 3 | grep -E "video|ALERT|fps_processed|unique_tracks|final_groups|done"
print(NAMES)

### 8b. Watch them

In [ ]:
import base64, json, pandas as pd
from IPython.display import HTML, display
def h264(mp4):
    out = mp4.replace('.mp4', '_h264.mp4')
    if not os.path.exists(out):
        !ffmpeg -y -loglevel error -i "{mp4}" -vcodec libx264 -pix_fmt yuv420p -crf 23 "{out}"
    return out
def show(mp4, width=1000):
    b = base64.b64encode(open(h264(mp4), 'rb').read()).decode()
    return HTML(f'<video width={width} controls><source src="data:video/mp4;base64,{b}" type="video/mp4"></video>')
for NAME in NAMES:
    print('=' * 20, NAME)
    display(show(f'{OUT}/{NAME}/full/annotated.mp4'))
    display(pd.read_csv(f'{OUT}/{NAME}/full/victims.csv').head(8))
    print(json.dumps(json.load(open(f'{OUT}/{NAME}/full/alerts.json'))[:5], indent=1))

## 9. Ablations (planned comparisons 3 and 4) on the main clip

Same clip, one module off at a time: does the ranking / do the alerts change? Pick the clip
with the most people as `MAIN`.

In [ ]:
MAIN = NAMES[0]          # <- change to the best clip's name
VIDEO = [v for v in vids if os.path.splitext(os.path.basename(v))[0] == MAIN][0]
for tag, flags in [('no_memory', '--no-memory'), ('no_severity', '--no-severity'),
                  ('no_motion', '--no-motion')]:
    print('=====', tag)
    !python run_rescue.py --source "{VIDEO}" --out "{OUT}/{MAIN}/{tag}" --detector {DET} --severity {SEV}         --stride 3 {flags} --no-panel | grep -E "fps_processed|unique_tracks|alerts|final_groups"
rows = []
for tag in ['full', 'no_memory', 'no_severity', 'no_motion']:
    p = f'{OUT}/{MAIN}/{tag}'
    s = json.load(open(f'{p}/summary.json'))
    v = pd.read_csv(f'{p}/victims.csv') if os.path.getsize(f'{p}/victims.csv') else pd.DataFrame()
    rows.append(dict(variant=tag, fps=s['fps_processed'], tracks=s.get('unique_tracks'),
                     alerts=s['alerts'], groups=s['final_groups'],
                     top_group=None if v.empty else f"{v.iloc[0]['group']} ({v.iloc[0]['people']} ppl)",
                     top_risk=None if v.empty else v.iloc[0]['risk'], **s['final_levels']))
abl = pd.DataFrame(rows); display(abl)
abl.to_csv(f'{OUT}/{MAIN}/ablation.csv', index=False)

## 10. Presentation pack - keyframes, tables, videos in one zip

Saves 4 keyframes per clip (PNG, for slides), the H.264 videos (play in PowerPoint), the
victim tables, alerts and the ablation table, then downloads `presentation_pack.zip`.

In [ ]:
import cv2, shutil
PACK = '/content/presentation_pack'
shutil.rmtree(PACK, ignore_errors=True); os.makedirs(PACK)
for NAME in NAMES:
    d = f'{OUT}/{NAME}/full'
    shutil.copy(h264(f'{d}/annotated.mp4'), f'{PACK}/{NAME}_demo.mp4')
    for f in ['victims.csv', 'alerts.json', 'summary.json']:
        shutil.copy(f'{d}/{f}', f'{PACK}/{NAME}_{f}')
    cap = cv2.VideoCapture(f'{d}/annotated.mp4'); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for k, q in enumerate([0.1, 0.4, 0.7, 0.98]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(q * (n - 1))); ok, fr = cap.read()
        if ok: cv2.imwrite(f'{PACK}/{NAME}_frame{k + 1}.png', fr)
if os.path.exists(f'{OUT}/{MAIN}/ablation.csv'):
    shutil.copy(f'{OUT}/{MAIN}/ablation.csv', f'{PACK}/{MAIN}_ablation.csv')
for p in sorted(glob.glob(f'{OUT}/test_photos/annotated/*.jpg'))[:12]:
    os.makedirs(f'{PACK}/photos', exist_ok=True); shutil.copy(p, f'{PACK}/photos/')
shutil.make_archive('/content/presentation_pack', 'zip', PACK)
shutil.copy('/content/presentation_pack.zip', f'{OUT}/presentation_pack.zip')
print(sorted(os.listdir(PACK)))
from google.colab import files
files.download('/content/presentation_pack.zip')